# Causal feature construction and calibration boundaries

Run the pipeline and dashboard export first. See [methodology](../docs/methodology.md) for protocol v2.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype'
ROOT = Path.cwd() if (Path.cwd()/'results').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT/'src'))
experiment = json.loads((ROOT/'results/experiment.json').read_text())
assert experiment['protocol_version'] == 2


## Inspect the measured feature distributions

In [2]:
features = pd.read_parquet(ROOT/'data/processed/features.parquet')
display(features[['pressure:n1|residual','flow:p227|residual','flow:p235|difference_1h']].describe())
pd.read_csv(ROOT/'results/feature_distributions.csv')

,pressure:n1|residual,flow:p227|residual,flow:p235|difference_1h
count,69599.000000,69599.000000,70075.000000
mean,-0.000259,-0.924577,0.003893
std,0.404606,11.096475,12.384464
min,-2.396667,-62.696667,-47.960000
25%,-0.183333,-4.846667,-7.570000
50%,-0.003333,0.406667,-1.256667
75%,0.173333,4.913333,5.026667
max,1.486667,40.463333,71.773333


,feature,count,mean,std,min,5%,50%,95%,max,split
0,pressure:n1|residual,9305.0,-0.004508,0.253631,-2.396667,-0.353333,0.003333,0.360000,1.253333,train
1,flow:p227|residual,9305.0,-0.947972,9.750958,-48.436667,-21.471333,0.136667,12.952667,35.323333,train
2,pressure:n1|std_1h,9305.0,0.054524,0.036287,0.000000,0.016639,0.047493,0.123486,0.401011,train
3,flow:p235|difference_1h,9305.0,-0.014605,12.755841,-33.090000,-17.644667,-1.350000,25.426000,71.773333,train
4,pressure:n1|residual,35040.0,0.006019,0.504845,-1.996667,-0.803333,-0.003333,0.826667,1.486667,test
5,flow:p227|residual,35040.0,-1.104565,11.740831,-60.180000,-22.566667,0.703333,16.863833,40.463333,test
6,pressure:n1|std_1h,35040.0,0.048137,0.035022,0.000000,0.012874,0.040127,0.116568,0.432164,test
7,flow:p235|difference_1h,35040.0,0.004742,12.155925,-47.960000,-16.376833,-1.306667,25.071167,66.246667,test


## Prove future measurements cannot change earlier features

In [3]:
from waterleak.features import causal_features
measurements = pd.read_parquet(ROOT/'data/processed/model_measurements.parquet').iloc[:1000]
before, _ = causal_features(measurements)
altered = measurements.copy()
altered.iloc[850:] = 10000
after, _ = causal_features(altered)
pd.testing.assert_frame_equal(before.iloc[:850], after.iloc[:850])
print('PASS: future changes leave all earlier features unchanged')

PASS: future changes leave all earlier features unchanged


## Prove estimator fitting and threshold reference are separated in time

In [4]:
fit = features.index[experiment['estimator_fit_row_positions']]
reference = features.index[experiment['threshold_reference_row_positions']]
assert fit.intersection(reference).empty and fit.max() < reference.min()
assert reference.max() < pd.Timestamp('2018-10-01')
pd.DataFrame({'start':[fit.min(),reference.min()], 'end':[fit.max(),reference.max()], 'rows':[len(fit),len(reference)]}, index=['Estimator fitting','Threshold reference'])

,start,end,rows
Estimator fitting,2018-01-15 13:45:00,2018-09-11 14:30:00,6000
Threshold reference,2018-09-11 14:45:00,2018-09-30 23:45:00,1861


## Check an unseen-reference threshold against its saved percentile

In [5]:
scores = pd.read_parquet(ROOT/'data/processed/scores.parquet')
method = 'Local Outlier Factor'
setting = experiment['thresholds'][method]
calculated = scores.loc[reference, method].quantile(setting['quantile'])
assert abs(calculated - setting['threshold']) < 1e-10
print('LOF threshold:', calculated, '; reference rows:', len(reference))

LOF threshold: 1.9356903314590554 ; reference rows: 1861


## Interpretation and limits

The trailing reference uses past measurements and can absorb slow leaks. All fitted transforms use the estimator subset only. LOF novelty scores for fitted rows must not calibrate thresholds. Fitting-pool states may already contain background leaks.